In [ ]:
import os
import sys
import glob

import cmocean
import lmdb
import numpy as np
import torch
import matplotlib.pyplot as plt

repo_root = os.environ.get("GAP_AWARE_ROOT", os.path.abspath("../../.."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from model_gap_aware.maskfeat import MaskFeatGapFiller

In [ ]:
TARGET_CHANNELS = [0]
_esdc_data = os.environ.get(
    "ESDC_DATA_DIR",
    os.path.join(repo_root, "esdc", "data", "lst_no_syn_gaps"),
)
LMDB_PATH = os.path.join(_esdc_data, "lmdb", "dataset.lmdb")
LOG_DIR = os.path.join(repo_root, "esdc", "lightning_logs", "experiments", "exp01_zero_shot", "version_4")
VAL_RANGE = (180.0, 360.0)  # Kelvin
CMAP = cmocean.cm.matter_r
VMIN, VMAX = 243.15, 323.15

N_SAMPLES = 5
N_TIMESTEPS = 5
TARGET_T = N_TIMESTEPS // 2
SEED = None

In [ ]:
ckpt_files = sorted(glob.glob(f"{LOG_DIR}/**/checkpoints/*.ckpt", recursive=True), key=os.path.getmtime)
if not ckpt_files:
    raise FileNotFoundError(f"No checkpoints found under {LOG_DIR} — run training first.")
print("Available checkpoints:")
for p in ckpt_files:
    print(f"  {p}")

ckpt_path = ckpt_files[-1]  # most recent; override manually if needed
print(f"\nUsing: {ckpt_path}")

model = MaskFeatGapFiller.load_from_checkpoint(ckpt_path, map_location="cpu")
model.eval()
print("Model loaded.")
print("hparams:", dict(model.hparams))
PATCH_SIZE = model.patch_size

# The checkpoint must have been trained on data with the same channel count as the LMDB.
with lmdb.open(LMDB_PATH, readonly=True, lock=False, subdir=False) as _env, _env.begin() as _txn:
    _d = _txn.get(b"0")
_ndim = int(np.frombuffer(_d, dtype=np.int32, count=1)[0])
data_channels = int(np.frombuffer(_d, dtype=np.int32, count=_ndim, offset=4)[1])
if model.hparams["in_channels"] != data_channels:
    raise ValueError(
        f"Checkpoint expects in_channels={model.hparams['in_channels']} but the LMDB has "
        f"{data_channels} channels. The checkpoint is stale — point LOG_DIR to a run trained on this dataset."
    )

In [ ]:
try:
    if "env" in globals() and env is not None:
        env.close()
except Exception:
    pass

env = lmdb.open(
    LMDB_PATH,
    readonly=True,
    lock=False,
    readahead=False,
    meminit=False,
    subdir=False,
)
n_total = env.stat()["entries"]
if n_total == 0:
    raise RuntimeError(f"LMDB is empty: {LMDB_PATH}")
print(f"LMDB entries: {n_total}")


def _decode(data: bytes) -> np.ndarray:
    ndim = int(np.frombuffer(data, dtype=np.int32, count=1, offset=0)[0])
    shape = tuple(np.frombuffer(data, dtype=np.int32, count=ndim, offset=4).tolist())
    header_nbytes = 4 * (1 + ndim)
    return np.frombuffer(data, dtype=np.float32, offset=header_nbytes).reshape(shape)


def load_sample(idx: int):
    with env.begin(write=False) as txn:
        data = txn.get(str(idx).encode())
    if data is None:
        raise IndexError(f"No entry for index {idx}")
    arr = _decode(data)                  # (T, C, H, W)
    x = torch.from_numpy(arr.copy())    # (T, C, H, W)
    mid = x.shape[0] // 2
    y = x[mid][TARGET_CHANNELS]         # (1, H, W)
    return x, y

In [ ]:
def compute_gap_patch_mask(frame, patch_size):
    """frame: (C, H, W) or (H, W) — NaN at gaps. Returns (H, W) bool, True = patch masked."""
    if frame.ndim == 2:
        frame = frame[np.newaxis]
    C, H, W = frame.shape
    n_ph, n_pw = H // patch_size, W // patch_size
    r = frame.reshape(C, n_ph, patch_size, n_pw, patch_size)
    patch_has_nan = np.isnan(r).any(axis=(0, 2, 4))
    return np.repeat(np.repeat(patch_has_nan, patch_size, axis=0), patch_size, axis=1).astype(bool)


def patch_mask_to_pixels(patch_mask_1d, H, patch_size):
    """Convert flat patch mask (N,) to pixel-space (H, H). Assumes square image."""
    n_p = H // patch_size
    return np.repeat(np.repeat(patch_mask_1d.reshape(n_p, n_p), patch_size, axis=0), patch_size, axis=1).astype(bool)

In [ ]:
rng = np.random.default_rng(SEED)
chosen_indices = sorted(rng.choice(n_total, N_SAMPLES, replace=False).tolist())
print(f"Selected indices: {chosen_indices}")

In [ ]:
N_COLS = N_TIMESTEPS + 3  # input frames | Target | Reconstruction | Error
fig, axes = plt.subplots(2 * N_SAMPLES, N_COLS, figsize=(3.2 * N_COLS, 3.5 * 2 * N_SAMPLES), squeeze=False)

masked_cmap = CMAP.copy()
# masked_cmap.set_bad("lightgray")

for i, idx in enumerate(chosen_indices):
    row_in   = 2 * i
    row_mask = 2 * i + 1

    x, y = load_sample(idx)   # x:(T,C,H,W)  y:(1,H,W)
    arr = x.numpy()
    lst = arr[:, 0, :, :].astype(np.float32)                          # (T, H, W)
    lst_k = lst * (VAL_RANGE[1] - VAL_RANGE[0]) + VAL_RANGE[0]       # → Kelvin

    with torch.no_grad():
        preds, patch_mask = model(x.unsqueeze(0))                                    # (1,1,H,W), (1,N)
        loss_val, loss_mask = model.compute_loss(preds, y.unsqueeze(0), patch_mask)  # scalar, (1,1,H,W)
        loss_rescaled, _ = model.rescaled_loss(preds, y.unsqueeze(0), patch_mask, VAL_RANGE)

    pred_np  = preds[0, 0].numpy()        # (H, W), normalized
    y_np     = y[0].numpy()               # (H, W), normalized
    lmask_np = loss_mask[0, 0].numpy()    # (H, W)

    pred_k = pred_np * (VAL_RANGE[1] - VAL_RANGE[0]) + VAL_RANGE[0]
    y_k    = np.nan_to_num(y_np) * (VAL_RANGE[1] - VAL_RANGE[0]) + VAL_RANGE[0]

    H, W = lst_k[0].shape

    # --- Input row: original frames ---
    for t in range(N_TIMESTEPS):
        ax = axes[row_in][t]
        ax.matshow(lst_k[t], cmap=CMAP, vmin=VMIN, vmax=VMAX)
        frac = 100.0 * float(np.isnan(lst[t]).sum()) / float(lst[t].size)
        title = f"t={t}  gaps={frac:.1f}%"
        if t == TARGET_T:
            title += "\n(target frame)"
        ax.set_title(title, fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])

    ax = axes[row_in][N_TIMESTEPS]
    ax.matshow(y_k, cmap=CMAP, vmin=VMIN, vmax=VMAX)
    ax.set_title(f"Target (idx={idx})", fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])

    ax = axes[row_in][N_TIMESTEPS + 1]
    blended_k = np.where(lmask_np > 0, pred_k, y_k)
    ax.matshow(blended_k, cmap=CMAP, vmin=VMIN, vmax=VMAX)
    ax.set_title("Reconstruction\n(pred in mask, GT outside)", fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])

    ax = axes[row_in][N_TIMESTEPS + 2]
    err_k = np.abs(pred_k - y_k)
    err_display = np.where(lmask_np > 0, err_k, np.nan)
    im = ax.imshow(err_display, cmap=cmocean.cm.amp, vmin=0)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="K")
    ax.set_title(f"Error\n{model.loss_type.upper()}={loss_rescaled.item():.2f} K", fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])

    # --- Masked-view row ---
    for t in range(N_TIMESTEPS):
        ax = axes[row_mask][t]
        if t == TARGET_T:
            # Middle frame: gap mask + random mask from the model forward pass
            pm_np = patch_mask[0].cpu().numpy()  # (N,)
            pixel_mask = patch_mask_to_pixels(pm_np, H, PATCH_SIZE)
        else:
            pixel_mask = compute_gap_patch_mask(lst[t], PATCH_SIZE)
        frame_masked = lst_k[t].copy()
        frame_masked[pixel_mask] = np.nan
        ax.matshow(frame_masked, cmap=masked_cmap, vmin=VMIN, vmax=VMAX)
        pct = 100.0 * pixel_mask.mean()
        title = f"t={t}  masked={pct:.1f}%"
        if t == TARGET_T:
            title += "\n(gap+rand)"
        ax.set_title(title, fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])
    for col in range(N_TIMESTEPS, N_COLS):
        axes[row_mask][col].axis("off")

plt.suptitle("Reconstruction Visualization — ESDC (LST)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()